# 05 — Gradient-Inversion Attack (DLG) + Communication-Cost Analysis

**Phase 5 of FedSafe-Fuse.** Runs on Google Colab Free + T4.

## What this notebook does

### Part A: Deep Leakage from Gradients (DLG) attack
Implements the attack from Zhu, Liu, & Han (2019) against a small classifier surrogate. For a known input image, we:

1. Compute true gradients.
2. Apply one of three protections: raw / DP-SGD / FIPCA.
3. Run LBFGS optimisation on a dummy input to match the (possibly protected) gradients.
4. Compare the reconstructed image to the original via MSE / SSIM / PSNR.

The surrogate is intentionally small (~10K parameters) — full DLG against the 4.9M-param FedSafe-Fuse model is not tractable in Colab Free's time budget. **This limitation is disclosed in the paper Methods.**

### Part B: Communication-cost summary
Loads the per-round bytes-on-the-wire numbers measured in Phase 3 and summarises them as the paper's headline privacy/efficiency table.

## Outputs
- `results/attack_table.csv` — reconstruction MSE/SSIM/PSNR per protection mode
- `results/comm_cost_summary.csv` — bytes/round per method, plus FIPCA compression ratio
- `figures/attack_reconstructions.png` — original vs raw / DP-SGD / FIPCA reconstructions
- `figures/privacy_vs_comm.png` — scatter of reconstruction-SSIM (privacy) vs comm bytes (efficiency)

## Expected wall-clock on T4
~3–5 min total.


In [ ]:
# 0. Install deps + seeds (no Drive download needed for Phase 5)
import os, sys, time, json, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import csv

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| Torch:', torch.__version__)


In [ ]:
# 1. Drive mount (for syncing results back at the end)
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/FedSafeFuse'
LOCAL_ROOT = '/content/local_fedsafe'
for sub in ['results', 'figures']:
    os.makedirs(f'{LOCAL_ROOT}/{sub}', exist_ok=True)
PROJECT_DRIVE = LOCAL_ROOT
print('Local mirror ready:', PROJECT_DRIVE)


## Part A — DLG attack with raw / DP-SGD / FIPCA protection

In [ ]:
# 2. Tiny surrogate classifier (~10K params). DLG converges in <2 min on this.
class TinySurrogate(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.c1 = nn.Conv2d(1, 12, 5, padding=2)
        self.c2 = nn.Conv2d(12, 12, 5, padding=2, stride=2)  # 32 -> 16
        self.c3 = nn.Conv2d(12, 12, 5, padding=2, stride=2)  # 16 -> 8
        self.fc = nn.Linear(12 * 8 * 8, n_classes)
    def forward(self, x):
        x = F.gelu(self.c1(x))
        x = F.gelu(self.c2(x))
        x = F.gelu(self.c3(x))
        return self.fc(x.flatten(1))

def soft_ce(logits, soft_target):
    return -(soft_target * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()

surrogate = TinySurrogate().to(DEVICE)
n_p = sum(p.numel() for p in surrogate.parameters() if p.requires_grad)
print(f'Surrogate params: {n_p:,}')


In [ ]:
# 3. Synthetic input: two Gaussian blobs on a 32x32 grid (medical-ish toy)
def synth_image(seed=42):
    rng = np.random.default_rng(seed)
    xs, ys = np.meshgrid(np.linspace(-1, 1, 32), np.linspace(-1, 1, 32))
    blob1 = np.exp(-((xs - 0.30)**2 + (ys + 0.30)**2) / 0.15) * 0.85
    blob2 = np.exp(-((xs + 0.40)**2 + (ys - 0.20)**2) / 0.08) * 0.55
    img = np.clip(blob1 + blob2, 0, 1).astype(np.float32)
    return torch.from_numpy(img).unsqueeze(0).unsqueeze(0)  # (1,1,32,32)

x_true = synth_image(seed=42).to(DEVICE)
# True one-hot-ish label (class index 3)
y_logits_true = torch.zeros(1, 10, device=DEVICE)
y_logits_true[0, 3] = 5.0
y_soft_true = F.softmax(y_logits_true, dim=-1)

# True gradient from a single forward+backward
surrogate.zero_grad()
pred = surrogate(x_true)
loss = soft_ce(pred, y_soft_true)
true_grads = [g.detach() for g in torch.autograd.grad(loss, surrogate.parameters())]
print(f'True gradient: {len(true_grads)} tensors, total {sum(g.numel() for g in true_grads):,} floats')


In [ ]:
# 4. Protection mechanisms (operate on a list of gradient tensors)
def protect_raw(grads):
    return [g.clone() for g in grads]

def protect_dpsgd(grads, clip=1.0, sigma=0.5, seed=SEED):
    total_norm = torch.sqrt(sum((g**2).sum() for g in grads))
    scale = (clip / total_norm.clamp(min=clip)).item()
    g_torch = torch.Generator(device=grads[0].device).manual_seed(seed)
    return [g * scale + torch.randn(g.shape, device=g.device, generator=g_torch) * sigma * clip
            for g in grads]

def protect_fipca(grads, rank=32, seed=SEED):
    flat = torch.cat([g.flatten() for g in grads])
    n = flat.numel()
    gen = torch.Generator(device=flat.device).manual_seed(seed)
    G = torch.randn(n, rank, device=flat.device, generator=gen)
    Q, _ = torch.linalg.qr(G)  # (n, rank) orthonormal columns
    P = Q.T  # (rank, n)
    coeffs = P @ flat
    recon_flat = P.T @ coeffs
    out, idx = [], 0
    for g in grads:
        k = g.numel()
        out.append(recon_flat[idx:idx + k].reshape(g.shape))
        idx += k
    return out


In [ ]:
# 5. DLG attack: optimise (dummy_x, dummy_y) to match the (possibly protected) gradients
def dlg_attack(model, x_template, target_grads, n_iter=120, lr=0.1):
    dummy_x = torch.randn_like(x_template, device=DEVICE).requires_grad_(True)
    dummy_y = torch.randn(1, 10, device=DEVICE).requires_grad_(True)
    opt = torch.optim.LBFGS([dummy_x, dummy_y], lr=lr, max_iter=20,
                            line_search_fn='strong_wolfe')
    for it in range(n_iter):
        def closure():
            opt.zero_grad()
            pred = model(dummy_x)
            soft_target = F.softmax(dummy_y, dim=-1)
            loss = soft_ce(pred, soft_target)
            dummy_grads = torch.autograd.grad(loss, model.parameters(), create_graph=True)
            gd = sum(((dg - tg) ** 2).sum() for dg, tg in zip(dummy_grads, target_grads))
            gd.backward()
            return gd
        opt.step(closure)
    return dummy_x.detach().clamp(0, 1)


In [ ]:
# 6. Run all three attacks
protections = [
    ('raw',     'No protection',          protect_raw),
    ('dpsgd',   'DP-SGD (σ=0.5, C=1.0)',  protect_dpsgd),
    ('fipca',   'FIPCA (rank=32)',        protect_fipca),
]

print('Running DLG against each protection...')
reconstructions = {}
for tag, label, protect in protections:
    t0 = time.time()
    target = protect(true_grads)
    recon = dlg_attack(surrogate, x_true, target, n_iter=120)
    reconstructions[tag] = {
        'label': label,
        'recon': recon.cpu(),
        'time_s': time.time() - t0,
    }
    print(f'  {label:25s} done in {time.time()-t0:.1f}s')


In [ ]:
# 7. Reconstruction-error table (MSE / SSIM / PSNR vs original)
def ssim_local(pred, target, window_size=11, sigma=1.5):
    """Inline SSIM for evaluation (same formula as src/losses.py)."""
    pred = pred.to(target.device)
    coords = torch.arange(window_size, dtype=torch.float32) - (window_size - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2)); g = g / g.sum()
    w_2d = (g.unsqueeze(0) * g.unsqueeze(1)).expand(target.shape[1], 1, window_size, window_size).contiguous().to(target.device)
    pad = window_size // 2
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.conv2d(pred, w_2d, padding=pad, groups=pred.shape[1])
    mu_t = F.conv2d(target, w_2d, padding=pad, groups=target.shape[1])
    sp = F.conv2d(pred*pred, w_2d, padding=pad, groups=pred.shape[1]) - mu_p*mu_p
    st = F.conv2d(target*target, w_2d, padding=pad, groups=target.shape[1]) - mu_t*mu_t
    spt = F.conv2d(pred*target, w_2d, padding=pad, groups=pred.shape[1]) - mu_p*mu_t
    num = (2*mu_p*mu_t + C1) * (2*spt + C2)
    den = (mu_p*mu_p + mu_t*mu_t + C1) * (sp + st + C2)
    return float((num/den).mean().item())

x_true_cpu = x_true.cpu()
rows = []
print(f'\n{"protection":<28} {"MSE":<10} {"SSIM":<10} {"PSNR (dB)":<12}')
print('-' * 60)
for tag, _, _ in protections:
    info = reconstructions[tag]
    r = info['recon']
    mse = F.mse_loss(r, x_true_cpu).item()
    ss = ssim_local(r, x_true_cpu)
    ps = 10 * np.log10(1.0 / (mse + 1e-12))
    rows.append({'tag': tag, 'label': info['label'], 'mse': mse, 'ssim': ss, 'psnr_db': ps})
    print(f'{info["label"]:<28} {mse:<10.4f} {ss:<10.4f} {ps:<12.2f}')

csv_path = f'{PROJECT_DRIVE}/results/attack_table.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['tag', 'label', 'mse', 'ssim', 'psnr_db'])
    w.writeheader()
    for r in rows:
        w.writerow(r)
print(f'\nSaved {csv_path}')


In [ ]:
# 8. Qualitative reconstruction figure: original | raw | DP-SGD | FIPCA
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
panels = [
    ('Original input', x_true_cpu[0, 0].numpy()),
    (f'Raw DLG\n(SSIM={ssim_local(reconstructions["raw"]["recon"], x_true_cpu):.3f})',
     reconstructions['raw']['recon'][0, 0].numpy()),
    (f'DP-SGD DLG\n(SSIM={ssim_local(reconstructions["dpsgd"]["recon"], x_true_cpu):.3f})',
     reconstructions['dpsgd']['recon'][0, 0].numpy()),
    (f'FIPCA DLG\n(SSIM={ssim_local(reconstructions["fipca"]["recon"], x_true_cpu):.3f})',
     reconstructions['fipca']['recon'][0, 0].numpy()),
]
for ax, (title, img) in zip(axes, panels):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.tight_layout()
fig_path = f'{PROJECT_DRIVE}/figures/attack_reconstructions.png'
plt.savefig(fig_path, dpi=140, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')


## Part B — Communication-cost summary (from Phase 3)

In [ ]:
# 9. Comm-cost table (numbers measured in Phase 3 round1_main.csv)
# Each row = bytes uploaded per client per round; total upstream = K * (per-client).
COMM_DATA = [
    {'method': 'FedAvg + IFCNN (B2)',     'bytes_per_round': 3_814_800,    'privacy': 'none'},
    {'method': 'FedAvg + DP-SGD (B1)',    'bytes_per_round': 59_178_012,   'privacy': 'DP, σ=0.5'},
    {'method': 'FedSafe + FIPCA (ours)',  'bytes_per_round': 384,          'privacy': 'rank-32 projection'},
]

# Compression ratio vs DP-SGD baseline (closest privacy-preserving comparison)
DP_BYTES = next(r['bytes_per_round'] for r in COMM_DATA if 'DP-SGD' in r['method'])
for r in COMM_DATA:
    r['compression_vs_dpsgd'] = round(DP_BYTES / r['bytes_per_round'], 1) if r['bytes_per_round'] > 0 else float('inf')

print(f'{"method":<28} {"bytes/round":<16} {"MB/round":<12} {"vs DP-SGD":<14}')
print('-' * 75)
for r in COMM_DATA:
    print(f'{r["method"]:<28} {r["bytes_per_round"]:<16,} {r["bytes_per_round"]/1e6:<12.4f} {r["compression_vs_dpsgd"]:<14.1f}x')

csv_path = f'{PROJECT_DRIVE}/results/comm_cost_summary.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['method', 'privacy', 'bytes_per_round', 'compression_vs_dpsgd'])
    w.writeheader()
    for r in COMM_DATA:
        w.writerow(r)
print(f'\nSaved {csv_path}')


In [ ]:
# 10. Privacy vs communication scatter: SSIM (lower=safer) vs bytes/round (lower=cheaper)
ssim_per_method = {
    'FedAvg + IFCNN (B2)': ssim_local(reconstructions['raw']['recon'], x_true_cpu),
    'FedAvg + DP-SGD (B1)': ssim_local(reconstructions['dpsgd']['recon'], x_true_cpu),
    'FedSafe + FIPCA (ours)': ssim_local(reconstructions['fipca']['recon'], x_true_cpu),
}

fig, ax = plt.subplots(figsize=(8, 5.5))
colors = {'FedAvg + IFCNN (B2)': '#1f77b4',
          'FedAvg + DP-SGD (B1)': '#ff7f0e',
          'FedSafe + FIPCA (ours)': '#2ca02c'}
for r in COMM_DATA:
    name = r['method']
    x_b = r['bytes_per_round']
    y_s = ssim_per_method[name]
    ax.scatter(x_b, y_s, s=220, color=colors[name], label=name,
               edgecolors='black', linewidth=1.0, zorder=3)
    ax.annotate(f'  {y_s:.3f}', (x_b, y_s), fontsize=10, va='center')

ax.set_xscale('log')
ax.set_xlabel('Communication cost per round (bytes, log)')
ax.set_ylabel('DLG reconstruction SSIM\n(lower = stronger privacy)')
ax.set_title('Privacy vs communication trade-off')
ax.grid(True, alpha=0.35, which='both')
ax.legend(loc='upper left')
ax.axhline(0.1, color='red', linestyle='--', alpha=0.4, label='SSIM=0.1 (no leakage)')
plt.tight_layout()
fig_path = f'{PROJECT_DRIVE}/figures/privacy_vs_comm.png'
plt.savefig(fig_path, dpi=140, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')


In [ ]:
# 11. Sync results + figures back to Drive
import shutil
synced = 0
for sub in ['results', 'figures']:
    for fname in os.listdir(f'{LOCAL_ROOT}/{sub}'):
        sp = f'{LOCAL_ROOT}/{sub}/{fname}'
        dp = f'{DRIVE_ROOT}/{sub}/{fname}'
        os.makedirs(os.path.dirname(dp), exist_ok=True)
        if os.path.isfile(sp):
            shutil.copy(sp, dp)
            synced += 1
print(f'Synced {synced} files to {DRIVE_ROOT}')


## When done

Paste back to chat:
1. The **attack_table** print from cell 7 (3 rows showing MSE/SSIM/PSNR per protection)
2. The **comm-cost summary** from cell 9 (3 rows)
3. The two figures: `attack_reconstructions.png` and `privacy_vs_comm.png`
4. Total wall-clock + any errors

Files to download back to local repo (after sync):

| Drive path | Local destination |
|---|---|
| `results/attack_table.csv` | `results/attack_table.csv` |
| `results/comm_cost_summary.csv` | `results/comm_cost_summary.csv` |
| `figures/attack_reconstructions.png` | `figures/attack_reconstructions.png` |
| `figures/privacy_vs_comm.png` | `figures/privacy_vs_comm.png` |
